# Пак Silero v5_6_ru для RuVoice (iMeet)

Собирает `ruvoice-pack-ru56.zip`: модель Silero `v5_6_ru` в формате паков [RuVoice TTS](https://github.com/kost-t-human/ruvoice-tts). Пак ставится в уже установленный RuVoice без пересборки приложения, а iMeet использует его голоса через системный TTS Android.

Запускайте ячейки по порядку: **Среда выполнения → Выполнить все**. Займёт 10–20 минут, нужен только CPU.

После сборки:
1. Скопируйте `ruvoice-pack-ru56.zip` на телефон.
2. RuVoice → меню → **Дополнительные голоса…** → **Установить из файла…** → выберите zip.
3. Откройте iMeet: голоса `Silero v5_6_ru` появятся в Voice Lab и в разговоре.

Лицензия модели Silero — некоммерческая (CC BY-NC). Пак годится для внутренних тестов; для продукта нужна коммерческая лицензия от Silero.

In [ ]:
# 1. ExecuTorch той же версии, что рантайм в RuVoice (executorch-1.5.0-xnnpack.aar)
!pip -q install "executorch==1.5.0" --extra-index-url https://download.pytorch.org/whl/cpu
import torch, executorch
print('torch', torch.__version__)

In [ ]:
# 2. RuVoice и модели Silero (v5_5_ru нужна только для сравнения входов)
!rm -rf ruvoice-tts && git clone -q --depth 1 https://github.com/kost-t-human/ruvoice-tts
!curl -sSL -o ruvoice-tts/tools/v5_6_ru.pt https://models.silero.ai/models/tts/ru/v5_6_ru.pt
!curl -sSL -o ruvoice-tts/tools/v5_5_ru.pt https://models.silero.ai/models/tts/ru/v5_5_ru.pt
!ls -la ruvoice-tts/tools/*.pt

In [ ]:
# 3. Проверка совместимости v5_6_ru с форматом паков RuVoice (рассчитан на v5_5_ru)
import torch

def describe(path):
    imp = torch.package.PackageImporter(path)
    big = imp.load_pickle('tts_models', 'model')
    pk = big.packages[0]
    try:
        type2id = imp.import_module('multi_acc_v3_package').TYPE2ID
    except Exception as e:
        type2id = f'не найдено: {e}'
    return {
        'packages': len(big.packages),
        'models': len(pk.models),
        'speakers': pk.speaker_to_ids[0],
        'has_accentor': pk.accentor is not None,
        'forward_args': len(pk.models[0].forward.schema.arguments) - 1,
        'type2id': type2id,
        'symbols_same_as_v55': None,
        '_pk': pk,
    }

v55 = describe('ruvoice-tts/tools/v5_5_ru.pt')
v56 = describe('ruvoice-tts/tools/v5_6_ru.pt')
v56['symbols_same_as_v55'] = v56['_pk'].symbols == v55['_pk'].symbols
for name, d in (('v5_5_ru', v55), ('v5_6_ru', v56)):
    print(name, {k: v for k, v in d.items() if k != '_pk'})

assert v56['packages'] == 1 and v56['models'] == 1, 'Другая структура модели: пак не соберётся'
assert v56['has_accentor'], 'У v5_6_ru нет акцентора: нужна доработка export_pack.py'
assert v56['forward_args'] == v55['forward_args'], 'У v5_6_ru другая сигнатура forward: RuVoice её не вызовет'
if v56['type2id'] != v55['type2id']:
    print('ВНИМАНИЕ: типы предложений отличаются. RuVoice передаёт типы по таблице v5_5_ru, '
          'поэтому новые типы (например, восклицание) могут не использоваться.')
else:
    print('Типы предложений совпадают с v5_5_ru.')
if not v56['symbols_same_as_v55']:
    print('Символы отличаются: это нормально, они записываются в pack.json.')

In [ ]:
# 4. Добавляем пак ru56 в tools/export_pack.py
p = 'ruvoice-tts/tools/export_pack.py'
src = open(p, encoding='utf-8').read()
entry = ("    'ru56': ('v5_6_ru.pt', lambda s: True, True, 'Голоса Silero v5.6 (iMeet)', 'CC BY-NC 4.0',\n"
         "             'https://models.silero.ai/models/tts/ru/v5_6_ru.pt', 'xenia'),\n")
anchor = "    'cis_be': ("
assert anchor in src and "'ru56'" not in src
src = src.replace(anchor, entry + anchor, 1)
count_anchor = "{'cis_ru': 29, 'ru': 5,"
assert count_anchor in src
src = src.replace(count_anchor, "{'cis_ru': 29, 'ru': 5, 'ru56': len(speakers),", 1)
open(p, 'w', encoding='utf-8').write(src)
print('export_pack.py: пак ru56 добавлен')

In [ ]:
# 5. Сборка и проверка пака (скрипт сам сверяет выход lite-моделей с исходной)
!cd ruvoice-tts && python tools/export_pack.py ru56

In [ ]:
# 6. Скачать пак
from google.colab import files
files.download('ruvoice-tts/dist/ruvoice-pack-ru56.zip')